# 01 · Tabular pipeline (drill)

**Style B: blind-code drill.** Each section has a spec and an empty code cell. Write the code from memory,
run the check cell under it, then delete your code (or `git restore` the notebook) so it is ready
for next time. The checks only use the names in the spec, never your internals.
Reference implementation: `../solutions/drills/01_tabular_pipeline.py`.

**Traces to** `notes/ioai-task-patterns.md`: P1, P2, P5 · T25-ANTIQUE, T24-O-BOBAI (macro F1), T24-H-ML (ROC AUC).
Logistic regression is a syllabus "Both" item, so section 5 implements it from scratch.

Total suggested time: **55 min**.

In [ ]:
import numpy as np, pandas as pd, os
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, f1_score

## 1. Load and split ⏱ 8 min

- Load OpenML `titanic` version 1 as a DataFrame.
- Drop the leaking columns `boat`, `body`, `home.dest`.
- `X`: a DataFrame containing at least `pclass, sex, age, sibsp, parch, fare, embarked` (add engineered columns if you like).
- `y`: an **int** Series (0/1).
- `X_train, X_test, y_train, y_test`: a stratified 80/20 split with `random_state=0`.

In [ ]:
assert isinstance(X, pd.DataFrame) and not {"boat", "body", "home.dest"} & set(X.columns)
assert {"pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"} <= set(X.columns)
assert pd.api.types.is_integer_dtype(y) and set(y.unique()) == {0, 1}
assert len(X_test) == 262 and abs(y_train.mean() - y_test.mean()) < 0.005
print("section 1 ok")

## 2. Preprocessing and model pipeline ⏱ 10 min

`clf`: a fitted `Pipeline`. Its first step is a `ColumnTransformer` that:
- median-imputes and standardises the numeric columns,
- most-frequent-imputes and one-hot-encodes the categorical columns, without crashing on unseen categories.

The last step is `LogisticRegression`. Fit it on the training split.

In [ ]:
assert isinstance(clf, Pipeline)
auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
odd = X_test.head(3).copy()
odd["embarked"] = pd.Series(["Z"] * 3, index=odd.index, dtype=object)   # a port never seen in training
clf.predict(odd)                                   # must not raise
assert auc >= 0.80, auc
print(f"section 2 ok (test AUC {auc:.3f})")

## 3. Cross-validation helper ⏱ 8 min

`cv_auc(model, X, y) -> float`: the mean ROC AUC over `StratifiedKFold(5, shuffle=True, random_state=0)`.
It must **not** refit or modify the `model` you pass in.

In [ ]:
coef_before = clf[-1].coef_.copy()
got = cv_auc(clf, X_train, y_train)
want = cross_val_score(clone(clf), X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc").mean()
assert np.isclose(got, want), (got, want)
assert np.array_equal(coef_before, clf[-1].coef_), "cv_auc modified the fitted model"
print(f"section 3 ok (CV AUC {got:.3f})")

## 4. Hyperparameter search ⏱ 10 min

`search`: a fitted `GridSearchCV` or `RandomizedSearchCV` over a pipeline that ends in
`HistGradientBoostingClassifier`. Tune at least 2 hyperparameters and score with ROC AUC.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
assert isinstance(search.best_estimator_[-1], HistGradientBoostingClassifier)
assert len(search.best_params_) >= 2 and search.scoring == "roc_auc"
test_auc = roc_auc_score(y_test, search.predict_proba(X_test)[:, 1])
assert search.best_score_ > 0.82 and test_auc > 0.82, (search.best_score_, test_auc)
print(f"section 4 ok (CV {search.best_score_:.3f}, test {test_auc:.3f})")

## 5. Logistic regression from scratch ⏱ 12 min

In NumPy only:
- `fit_logreg(X, y, lam=1e-2, lr=0.1, epochs=500) -> (w, b, losses)`: full-batch gradient descent on the mean
  log-loss plus `(lam/2)·||w||²`. The bias is not regularised. `losses` is a list with one value per epoch.
- `predict_proba_np(X, w, b) -> p`: shape `(n,)`, values in [0, 1].

The check runs on standardised breast-cancer data.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
Xb, yb = load_breast_cancer(return_X_y=True)
Xa, Xc, ya, yc = train_test_split(Xb, yb, test_size=0.3, random_state=0, stratify=yb)
sc = StandardScaler().fit(Xa); Xa, Xc = sc.transform(Xa), sc.transform(Xc)
w, b, losses = fit_logreg(Xa, ya, lam=1e-2, lr=0.1, epochs=500)
p = predict_proba_np(Xc, w, b)
assert p.shape == (len(Xc),) and p.min() >= 0 and p.max() <= 1
assert len(losses) == 500 and losses[-1] < 0.5 * losses[0], "loss did not go down enough"
mine = ((p > 0.5) == yc).mean()
ref = LogisticRegression(C=1 / (1e-2 * len(Xa)), max_iter=5000).fit(Xa, ya).score(Xc, yc)
assert mine >= ref - 0.03, (mine, ref)
print(f"section 5 ok (acc {mine:.3f} vs sklearn {ref:.3f})")

## 6. Threshold for macro F1, and an IOAI-format file ⏱ 7 min

- `best_threshold(y_true, proba) -> float`: the value in `np.linspace(0.05, 0.95, 91)` that maximises macro F1.
- `write_submission(pred, path)`: one integer per line, no header, no index.

In [ ]:
proba = search.predict_proba(X_test)[:, 1]
ts = np.linspace(0.05, 0.95, 91)
want = max(f1_score(y_test, (proba >= t).astype(int), average="macro") for t in ts)
t = best_threshold(y_test, proba)
assert np.isclose(f1_score(y_test, (proba >= t).astype(int), average="macro"), want)
write_submission((proba >= t).astype(int), "check_sub.csv")
lines = open("check_sub.csv").read().splitlines()
assert len(lines) == len(y_test) and set(lines) <= {"0", "1"}
os.remove("check_sub.csv")
print(f"section 6 ok (t={t:.2f}, macro F1 {want:.3f})")